# House Prices: Data Preprocessing for ML

This notebook focuses on cleaning, preprocessing, and preparing data for machine learning models.

**Key Tasks:**
1. Handle missing values properly
2. Encode categorical and ordinal features
3. Scale numerical features
4. Create feature engineering pipeline
5. Prepare train/test splits for modeling

In [58]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

from src.utils import (
    load_data, identify_na_as_absence_columns, replace_na_with_none,
    numeric_vs_categorical_split
)

In [59]:
# Load raw data
train, test, sample = load_data("../data/")

print("Original training data shape:", train.shape)
print("Original test data shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)
Sample submission shape: (1459, 2)
Original training data shape: (1460, 81)
Original test data shape: (1459, 80)


## 1. Handle "Absence" NA Values

In [60]:
# Columns where NA means "absence of feature"
absence_cols = identify_na_as_absence_columns()
print(f"Columns to convert NA to 'None':\n{absence_cols}\n")

# Replace NA with 'None'
train_processed = replace_na_with_none(train, absence_cols)
test_processed = replace_na_with_none(test, absence_cols)

print("After processing:")
print(f"Training set - NA count: {train_processed.isnull().sum().sum()}")
print(f"Test set - NA count: {test_processed.isnull().sum().sum()}")

Columns to convert NA to 'None':
['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']

After processing:
Training set - NA count: 1221
Test set - NA count: 1236


## 2. Define Ordinal Feature Mappings

In [61]:
# Ordinal features and their mappings (Ex=5, Gd=4, TA=3, Fa=2, Po=1, None=0)
ordinal_features = {
    'ExterQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'ExterCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
    'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0},
    'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'FireplaceQu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'GarageQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'GarageCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'PoolQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0}
}

# Apply ordinal encoding
for feature, mapping in ordinal_features.items():
    if feature in train_processed.columns:
        train_processed[feature] = train_processed[feature].map(mapping).fillna(0)
        test_processed[feature] = test_processed[feature].map(mapping).fillna(0)

print("Ordinal encoding applied successfully")

Ordinal encoding applied successfully


## 2.1 Handle Numericals as Numericals

In [62]:
# MSSubClass is stored as int but should be treated as categorical
print("Before conversion:")
print(f"MSSubClass dtype: {train_processed['MSSubClass'].dtype}")
print(f"Unique values: {sorted(train_processed['MSSubClass'].unique())}")

# Convert MSSubClass to string/categorical
train_processed['MSSubClass'] = train_processed['MSSubClass'].astype(str)
test_processed['MSSubClass'] = test_processed['MSSubClass'].astype(str)

print("\nAfter conversion:")
print(f"MSSubClass dtype: {train_processed['MSSubClass'].dtype}")

Before conversion:
MSSubClass dtype: int64
Unique values: [np.int64(20), np.int64(30), np.int64(40), np.int64(45), np.int64(50), np.int64(60), np.int64(70), np.int64(75), np.int64(80), np.int64(85), np.int64(90), np.int64(120), np.int64(160), np.int64(180), np.int64(190)]

After conversion:
MSSubClass dtype: str


## 3. Handle Remaining Missing Values

In [63]:
# Identify remaining missing values
train_missing = train_processed.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)

test_missing = test_processed.isnull().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("Remaining missing values in training set:")
print(train_missing.to_string())
print("\nRemaining missing values in test set:")
print(test_missing.to_string())


Remaining missing values in training set:
MasVnrType     872
LotFrontage    259
GarageYrBlt     81
MasVnrArea       8
Electrical       1

Remaining missing values in test set:
MasVnrType      894
LotFrontage     227
GarageYrBlt      78
MasVnrArea       15
MSZoning          4
Utilities         2
BsmtHalfBath      2
Functional        2
BsmtFullBath      2
Exterior1st       1
BsmtUnfSF         1
BsmtFinSF2        1
BsmtFinSF1        1
Exterior2nd       1
TotalBsmtSF       1
GarageCars        1
GarageArea        1
SaleType          1


In [64]:
# Imputation strategy 
# Note (Bryce): This is the simplest imputation stategy. Obviously this is not perfect, and can alter the probability distribution of values in training.
#               This issue could be addressed in the future with conditional/reggressional/MI methods (Advance - have to refer to notes)
# Numerical: median imputation
# Categorical: most frequent imputation (mode)

numeric_cols, categorical_cols = numeric_vs_categorical_split(train_processed, exclude_cols=['Id', 'SalePrice'])

# Create imputers
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Apply imputation
train_processed[numeric_cols] = numeric_imputer.fit_transform(train_processed[numeric_cols])
train_processed[categorical_cols] = categorical_imputer.fit_transform(train_processed[categorical_cols])

test_processed[numeric_cols] = numeric_imputer.transform(test_processed[numeric_cols])
test_processed[categorical_cols] = categorical_imputer.transform(test_processed[categorical_cols])

# Verify no missing values remain
print(f"Missing values in training after imputation: {train_processed.isnull().sum().sum()}")
print(f"Missing values in test after imputation: {test_processed.isnull().sum().sum()}")

Missing values in training after imputation: 0
Missing values in test after imputation: 0


## 4. Feature Engineering

In [65]:
def engineer_features(df):
    """Add engineered features to the dataframe."""
    df_eng = df.copy()
    
    # ========== AREA FEATURES ==========
    # Total living area (above ground + basement)
    df_eng['TotalSF'] = df_eng['GrLivArea'] + df_eng['TotalBsmtSF']
    
    # Total bathrooms (full + half)
    df_eng['TotalBath'] = (df_eng['FullBath'] + 
                           0.5 * df_eng['HalfBath'] + 
                           df_eng['BsmtFullBath'] + 
                           0.5 * df_eng['BsmtHalfBath'])
    
    # Total porch area
    df_eng['TotalPorchSF'] = (df_eng['OpenPorchSF'] + 
                               df_eng['EnclosedPorch'] + 
                               df_eng['3SsnPorch'] + 
                               df_eng['ScreenPorch'])
    
    # ========== AGE FEATURES ==========
    current_year = 2010  # Last year in dataset
    df_eng['HouseAge'] = current_year - df_eng['YearBuilt']
    df_eng['RemodAge'] = current_year - df_eng['YearRemodAdd']
    df_eng['IsRemodeled'] = (df_eng['YearBuilt'] != df_eng['YearRemodAdd']).astype(int)
    
    # Special: Pre-1950 or on 1950 remodel flag (based on your observation)
    df_eng['IsOnOrBefore1950Remodel'] = (df_eng['YearRemodAdd'] <= 1950).astype(int)
    
    # Remodel era categories (more granular than binary)
    def get_remodel_era(year_remod, year_built):
        if year_remod <= 1950:
            return 'OnOrBefore_1950'
        elif year_remod == year_built:
            return 'Original_No_Remodel'
        elif year_remod < 2000:
            return 'Remodel_1951_1999'
        else:
            return 'Remodel_2000_Plus'
    
    df_eng['RemodelEra'] = df_eng.apply(
        lambda x: get_remodel_era(x['YearRemodAdd'], x['YearBuilt']), axis=1
    )
    
    # # ========== SEASONALITY FEATURES ========== >>> Removed as I dont believe this to be based on any information
    # # Month features (cyclical encoding)
    # df_eng['MoSold_sin'] = np.sin(2 * np.pi * df_eng['MoSold'] / 12)
    # df_eng['MoSold_cos'] = np.cos(2 * np.pi * df_eng['MoSold'] / 12)
    
    # # Quarter of year (1-4)
    # df_eng['Quarter'] = ((df_eng['MoSold'] - 1) // 3 + 1).astype(int)
    
    # # Peak season flag (spring/summer vs fall/winter)
    # # Based on typical real estate seasonality
    # df_eng['IsPeakSeason'] = df_eng['MoSold'].isin([5, 6, 7, 8]).astype(int)  # May-August peak
    
    # # Holiday effect (December holidays might affect sales)
    # df_eng['IsHolidayMonth'] = df_eng['MoSold'].isin([12]).astype(int)
    
    # # Year fixed effects (capture market trends)
    # # We'll keep YrSold as is, but could also create decade bins
    # df_eng['YearDecade'] = (df_eng['YrSold'] // 10) * 10
    
    # ========== OUTLIER DATE COMBINATIONS ==========
    # Create combined date identifier for potential outlier detection
    df_eng['YearMonth'] = df_eng['YrSold'].astype(str) + '_' + df_eng['MoSold'].astype(str)
    
    # Flag specific outlier date ranges (customize based on EDA findings)
    # Example: Financial crisis period (2008-2009)
    df_eng['IsFinancialCrisis'] = ((df_eng['YrSold'] >= 2008) & (df_eng['YrSold'] <= 2009)).astype(int)
    
    # Post-housing bubble period
    df_eng['IsPostBubble'] = (df_eng['YrSold'] >= 2007).astype(int)
    
    # ========== BINARY FLAGS (Presence features) ==========
    df_eng['HasBsmt'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)
    df_eng['HasPool'] = (df_eng['PoolArea'] > 0).astype(int)
    df_eng['HasFireplace'] = (df_eng['Fireplaces'] > 0).astype(int)
    
    # ========== QUALITY FEATURES ==========
    quality_cols = ['ExterQual', 'BsmtQual', 'HeatingQC', 'KitchenQual', 'GarageQual']
    df_eng['TotalQual'] = df_eng[quality_cols].sum(axis=1)
    
    # Quality interaction with age
    df_eng['QualPerAge'] = df_eng['TotalQual'] / (df_eng['HouseAge'] + 1)
    
    # ========== AREA PER ROOM ==========
    df_eng['AreaPerRoom'] = df_eng['GrLivArea'] / (df_eng['TotRmsAbvGrd'] + 1)
    
    # ========== LOT FEATURES ==========
    # Lot shape irregularity (treat as categorical but create binary flag)
    df_eng['IsIrregularLot'] = df_eng['LotShape'].isin(['IR1', 'IR2', 'IR3']).astype(int)
    
    # Land slope severity
    df_eng['IsSevereSlope'] = (df_eng['LandSlope'] == 'Sev').astype(int)


    # 25/05 added outlier catching
    # ========== OUTLIERS ==========
    # GrLivArea outlier flag (>4000 sq ft)
    df_eng['IsLargeGrLivArea'] = (df_eng['GrLivArea'] > 4000).astype(int)
    
    # TotalBsmtSF outlier flag (>3000 sq ft)
    df_eng['IsLargeTotalBsmtSF'] = (df_eng['TotalBsmtSF'] > 3000).astype(int)
        
    # 1stFlrSF outlier flag (>3000 sq ft)
    df_eng['IsLarge1stFlrSF'] = (df_eng['1stFlrSF'] > 3000).astype(int)

    return df_eng

In [66]:
# Apply feature engineering
train_processed = engineer_features(train_processed)
test_processed = engineer_features(test_processed)

print("Feature engineering complete")
print(f"New training shape: {train_processed.shape}")
print(f"New test shape: {test_processed.shape}")

Feature engineering complete
New training shape: (1460, 104)
New test shape: (1459, 103)


In [67]:
# Update numeric/categorical split after engineering
numeric_cols, categorical_cols = numeric_vs_categorical_split(train_processed, exclude_cols=['Id', 'SalePrice'])


# Manually add engineered categorical features that might be detected as numeric
engineered_categorical = ['RemodelEra', 'YearMonth']
# Convert any engineered categoricals that are numeric
for col in engineered_categorical:
    if col in train_processed.columns and col not in categorical_cols:
        categorical_cols.append(col)
        if col in numeric_cols:
            numeric_cols.remove(col)


# Also ensure MSSubClass is in categorical
if 'MSSubClass' in numeric_cols:
    numeric_cols.remove('MSSubClass')
    categorical_cols.append('MSSubClass')

print(f"Final numeric features: {len(numeric_cols)}")
print(f"Final categorical features: {len(categorical_cols)}")
print(f"\nFirst 20 categorical features: {categorical_cols[:20]}")

Final numeric features: 70
Final categorical features: 32

First 20 categorical features: ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation']


## 5. Handle Categorical Features

In [68]:
# Identify cardinality for all categorical features (including engineered)
cardinality = {}
for col in categorical_cols:
    if col in train_processed.columns:
        cardinality[col] = train_processed[col].nunique()

# Define thresholds
high_cardinality = [col for col, val in cardinality.items() if val > 10 and col != 'YearMonth']  # YearMonth excluded (too many unique)
#medium_cardinality = [col for col, val in cardinality.items() if 5 <= val <= 10]
low_cardinality = [col for col, val in cardinality.items() if 2 <= val <= 10]

print(f"High cardinality (10+ unique): {high_cardinality}")
print(f"Low cardinality (2-9 unique): {low_cardinality[:10]}...")
print(f"Special handling needed: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd'] - consider target encoding")

# For YearMonth, we can extract features rather than encoding all combinations
if 'YearMonth' in train_processed.columns:
    # Extract components instead of encoding
    train_processed['Year'] = train_processed['YrSold']
    test_processed['Year'] = test_processed['YrSold']
    train_processed['Month'] = train_processed['MoSold']
    test_processed['Month'] = test_processed['MoSold']
    
    # Drop the combined YearMonth column to avoid explosion of dummy variables
    train_processed = train_processed.drop('YearMonth', axis=1)
    test_processed = test_processed.drop('YearMonth', axis=1)
    
    # Update categorical columns
    categorical_cols = [col for col in categorical_cols if col != 'YearMonth']
    low_cardinality.extend(['Year', 'Month'])  # Add these as categorical


High cardinality (10+ unique): ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
Low cardinality (2-9 unique): ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2']...
Special handling needed: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd'] - consider target encoding


In [69]:
# Option 1: One-hot encoding for low cardinality features
# Option 2: Target encoding for high cardinality (will be done in modeling notebook)
# For now, we'll prepare both options

# One-hot encode low cardinality features
train_encoded = pd.get_dummies(train_processed, columns=low_cardinality, drop_first=True)
test_encoded = pd.get_dummies(test_processed, columns=low_cardinality, drop_first=True)

# Align columns (test might have missing categories)
train_cols = set(train_encoded.columns)
test_cols = set(test_encoded.columns)

# Add missing columns to test
for col in train_cols - test_cols:
    if col != 'SalePrice':
        test_encoded[col] = 0

# Remove extra columns from test
for col in test_cols - train_cols:
    if col != 'Id':
        test_encoded = test_encoded.drop(columns=[col])

print(f"Encoded training shape: {train_encoded.shape}")
print(f"Encoded test shape: {test_encoded.shape}")

Encoded training shape: (1460, 203)
Encoded test shape: (1459, 202)


## 6. Scale Numerical Features

In [70]:
# Identify numerical columns (after encoding)
encoded_numeric_cols = train_encoded.select_dtypes(include=[np.number]).columns.tolist()
encoded_numeric_cols = [col for col in encoded_numeric_cols if col not in ['Id', 'SalePrice']]

print(f"Numerical features to scale: {len(encoded_numeric_cols)}")

# Use RobustScaler (less sensitive to outliers)
scaler = RobustScaler()

# Fit on training, transform both
train_encoded[encoded_numeric_cols] = scaler.fit_transform(train_encoded[encoded_numeric_cols])
test_encoded[encoded_numeric_cols] = scaler.transform(test_encoded[encoded_numeric_cols])

print("Feature scaling complete")

Numerical features to scale: 70
Feature scaling complete


## 7. Prepare Final Datasets for Modeling


In [71]:
# Log transform SalePrice
y_train = np.log1p(train_encoded['SalePrice'])
X_train = train_encoded.drop(['Id', 'SalePrice'], axis=1)
X_test = test_encoded.drop(['Id'], axis=1)

print(f"Features for modeling: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Save processed data for modeling
X_train.to_csv("../data/X_train_processed.csv", index=False)
y_train.to_csv("../data/y_train_processed.csv", index=False)
X_test.to_csv("../data/X_test_processed.csv", index=False)

print("\nProcessed data saved for modeling notebook")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")


Features for modeling: 201
Training samples: 1460
Test samples: 1459

Processed data saved for modeling notebook
X_train shape: (1460, 201)
y_train shape: (1460,)
X_test shape: (1459, 201)


In [72]:
# Quick validation split for initial testing
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"\nTrain/Validation split:")
print(f"Training: {X_train_split.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")

# Save split data
X_train_split.to_csv("../data/X_train_split.csv", index=False)
y_train_split.to_csv("../data/y_train_split.csv", index=False)


Train/Validation split:
Training: 1168 samples
Validation: 292 samples
